<a href="https://colab.research.google.com/github/marcocslima/dev/blob/master/LRC_X_Cifra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
from difflib import SequenceMatcher
import unicodedata

# --- CONFIGURAÇÕES DE EXPR. REGULARES ---
CHORD_REGEX = re.compile(
    r'^[A-G][#b]?(?:m|maj|min|dim|aug|sus|add|at)?(?:\d+)?(?:[#b]\d+)?(?:/[A-G][#b]?)?$'
)
HEADER_REGEX = re.compile(r'^(\[[^\]]+\])\s*(.*)$')
LRC_REGEX = re.compile(r'^\[(\d{2}:\d{2}(?:\.\d{2,3})?)\]\s*(.*)$')


# --- FUNÇÕES DE SUPORTE ---

def read_file(file_path):
    """Lê arquivos de texto testando codificações comuns para evitar erros de acentos."""
    for encoding in ['utf-8', 'latin-1', 'cp1252']:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return f.read()
        except (UnicodeDecodeError, FileNotFoundError):
            continue
    raise FileNotFoundError(f"Não foi possível abrir ou decodificar o arquivo: {file_path}")


def is_chord_line(line):
    tokens = line.split()
    if not tokens:
        return False
    cleaned_tokens = [t.strip('()[]{}') for t in tokens]
    chord_count = sum(1 for t in cleaned_tokens if CHORD_REGEX.match(t))
    return (chord_count / len(tokens)) >= 0.7


def get_chords_from_line(chord_line):
    chords = []
    for match in re.finditer(r'\S+', chord_line):
        chord = match.group()
        start_idx = match.start()
        chords.append((start_idx, chord))
    return chords


def parse_cifra(cifra_text):
    lines = cifra_text.split('\n')
    parsed_items = []
    pending_chord_line = None

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue

        header_match = HEADER_REGEX.match(stripped)
        header = None
        content = line
        if header_match:
            header = header_match.group(1)
            content_start = line.find(header_match.group(2))
            content = line[content_start:] if content_start != -1 else ""

        if is_chord_line(content):
            if pending_chord_line is not None:
                parsed_items.append({
                    'type': 'instrumental',
                    'chords': pending_chord_line,
                    'lyric': ''
                })
            pending_chord_line = content
        else:
            if header and not content.strip():
                if pending_chord_line is not None:
                    parsed_items.append({
                        'type': 'instrumental',
                        'chords': pending_chord_line,
                        'lyric': ''
                    })
                    pending_chord_line = None
                parsed_items.append({
                    'type': 'header',
                    'text': header
                })
            else:
                parsed_items.append({
                    'type': 'lyric',
                    'chords': pending_chord_line if pending_chord_line else "",
                    'lyric': line.strip()
                })
                pending_chord_line = None

    if pending_chord_line is not None:
        parsed_items.append({
            'type': 'instrumental',
            'chords': pending_chord_line,
            'lyric': ''
        })

    return parsed_items


def parse_lrc(lrc_text):
    lrc_lines = []
    for line in lrc_text.split('\n'):
        line = line.strip()
        if not line:
            continue
        match = LRC_REGEX.match(line)
        if match:
            lrc_lines.append({
                'timestamp': match.group(1),
                'text': match.group(2).strip()
            })
    return lrc_lines


def normalize_text(text):
    text = text.lower()
    text = "".join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    text = re.sub(r'[^\w\s]', '', text)
    return " ".join(text.split())


def map_index(src_text, tgt_text, src_idx):
    matcher = SequenceMatcher(None, src_text, tgt_text)
    matching_blocks = matcher.get_matching_blocks()

    for block in matching_blocks:
        src_start = block.a
        tgt_start = block.b
        size = block.size
        if src_start <= src_idx < src_start + size:
            offset = src_idx - src_start
            return tgt_start + offset

    closest_dist = float('inf')
    best_idx = src_idx
    for block in matching_blocks:
        if block.size == 0:
            continue
        src_start = block.a
        src_end = block.a + block.size
        tgt_start = block.b
        tgt_end = block.b + block.size

        d_start = abs(src_idx - src_start)
        if d_start < closest_dist:
            closest_dist = d_start
            best_idx = tgt_start
        d_end = abs(src_idx - src_end)
        if d_end < closest_dist:
            closest_dist = d_end
            best_idx = tgt_end

    return best_idx


def align_lrc_and_cifra(lrc_lines, cifra_lyrics):
    norm_lrc = [normalize_text(line['text']) for line in lrc_lines]
    norm_cifra = [normalize_text(item['lyric']) for item in cifra_lyrics]

    matcher = SequenceMatcher(None, norm_lrc, norm_cifra)
    matching_blocks = matcher.get_matching_blocks()

    lrc_to_cifra = {}
    for block in matching_blocks:
        for i in range(block.size):
            lrc_to_cifra[block.a + i] = block.b + i

    last_matched_cifra_idx = -1
    next_matched_cifra_idx = [len(cifra_lyrics)] * len(lrc_lines)
    current_next = len(cifra_lyrics)
    for i in range(len(lrc_lines) - 1, -1, -1):
        if i in lrc_to_cifra:
            current_next = lrc_to_cifra[i]
        next_matched_cifra_idx[i] = current_next

    for i in range(len(lrc_lines)):
        if i in lrc_to_cifra:
            last_matched_cifra_idx = lrc_to_cifra[i]
            continue

        search_start = max(0, last_matched_cifra_idx)
        search_end = min(len(cifra_lyrics), next_matched_cifra_idx[i] + 1)

        if search_end <= search_start:
            search_start = max(0, last_matched_cifra_idx - 3)
            search_end = min(len(cifra_lyrics), last_matched_cifra_idx + 4)

        best_score = 0.0
        best_cifra_idx = None

        for c_idx in range(search_start, search_end):
            if c_idx >= len(cifra_lyrics):
                continue
            score = SequenceMatcher(None, norm_lrc[i], norm_cifra[c_idx]).ratio()
            if score > best_score and score > 0.4:
                best_score = score
                best_cifra_idx = c_idx

        if best_cifra_idx is not None:
            lrc_to_cifra[i] = best_cifra_idx
            last_matched_cifra_idx = best_cifra_idx

    return lrc_to_cifra


def build_chord_line(mapped_chords):
    mapped_chords = sorted(mapped_chords, key=lambda x: x[0])
    chord_line_chars = []
    current_pos = 0
    for idx, chord in mapped_chords:
        if idx < current_pos:
            idx = current_pos + 1
        chord_line_chars.append(' ' * (idx - current_pos))
        chord_line_chars.append(chord)
        current_pos = idx + len(chord)
    return "".join(chord_line_chars)


def build_inline_line(lyric_text, mapped_chords):
    mapped_chords = sorted(mapped_chords, key=lambda x: x[0], reverse=True)
    chars = list(lyric_text)
    for idx, chord in mapped_chords:
        insert_pos = min(idx, len(chars))
        chars.insert(insert_pos, f"[{chord}]")
    return "".join(chars)


# --- FUNÇÃO PRINCIPAL DE PROCESSAMENTO ---

def mesclar_cifra_e_lrc(cifra_path, lrc_path, output_dir=None):
    """
    Recebe os caminhos de um arquivo de cifra e um arquivo LRC,
    alinha ambos e gera dois arquivos novos no formato LRC com os acordes.
    """
    try:
        cifra_text = read_file(cifra_path)
        lrc_text = read_file(lrc_path)
    except FileNotFoundError as e:
        print(f"[Erro] {e}")
        return None, None

    parsed_cifra = parse_cifra(cifra_text)
    parsed_lrc = parse_lrc(lrc_text)

    cifra_lyrics = [item for item in parsed_cifra if item['type'] == 'lyric']
    lrc_to_cifra_mapping = align_lrc_and_cifra(parsed_lrc, cifra_lyrics)

    output_two_lines = []
    output_inline = []

    for i, lrc_line in enumerate(parsed_lrc):
        timestamp = lrc_line['timestamp']
        lyric_text = lrc_line['text']

        if i in lrc_to_cifra_mapping:
            cifra_item = cifra_lyrics[lrc_to_cifra_mapping[i]]
            raw_chords = cifra_item['chords']
            src_lyric = cifra_item['lyric']

            if raw_chords:
                chords_list = get_chords_from_line(raw_chords)
                mapped_chords = []
                for orig_idx, chord_name in chords_list:
                    mapped_idx = map_index(src_lyric, lyric_text, orig_idx)
                    mapped_chords.append((mapped_idx, chord_name))

                # Formato de duas linhas
                chord_line_str = build_chord_line(mapped_chords)
                output_two_lines.append(f"[{timestamp}] {chord_line_str}")
                output_two_lines.append(f"[{timestamp}] {lyric_text}")

                # Formato inline
                inline_str = build_inline_line(lyric_text, mapped_chords)
                output_inline.append(f"[{timestamp}] {inline_str}")
            else:
                output_two_lines.append(f"[{timestamp}] {lyric_text}")
                output_inline.append(f"[{timestamp}] {lyric_text}")
        else:
            output_two_lines.append(f"[{timestamp}] {lyric_text}")
            output_inline.append(f"[{timestamp}] {lyric_text}")

    # Determinar pasta de saída
    if output_dir is None:
        output_dir = os.path.dirname(lrc_path) if os.path.dirname(lrc_path) else "/content"

    os.makedirs(output_dir, exist_ok=True)

    # Criar caminhos dos novos arquivos baseando-se no nome original do LRC
    base_name = os.path.splitext(os.path.basename(lrc_path))[0]
    out_two_lines_path = os.path.join(output_dir, f"{base_name}_duas_linhas.lrc")
    out_inline_path = os.path.join(output_dir, f"{base_name}_inline.lrc")

    # Salvar resultados
    with open(out_two_lines_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(output_two_lines))

    with open(out_inline_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(output_inline))

    print(f"\n[Sucesso] Arquivos gerados para: {base_name}")
    print(f"  -> Salvo (Duas Linhas): {out_two_lines_path}")
    print(f"  -> Salvo (Inline): {out_inline_path}")

    return out_two_lines_path, out_inline_path

In [ ]:
# Configure os caminhos exatos dos arquivos no Colab
cifra_input = '/content/cifra.txt'
lrc_input = '/content/lrc.txt'

# Executa o processamento
duas_linhas_file, inline_file = mesclar_cifra_e_lrc(cifra_input, lrc_input)

# Mostrar uma prévia rápida de como ficou um pedaço do resultado (opcional)
if duas_linhas_file:
    print("\n--- PRÉVIA DO ARQUIVO COM DUAS LINHAS (Primeiras 10 linhas) ---")
    with open(duas_linhas_file, 'r', encoding='utf-8') as f:
        print("".join(f.readlines()[:10]))